<a href="https://colab.research.google.com/github/cmont13/IDX_Winter26_DS40/blob/main/notebook04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.4 MB/s eta 0:00:00


In [87]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import category_encoders as ce
from sklearn.neighbors import KNeighborsRegressor

In [86]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Loading Data

In [97]:
property_dfs = []

dir_path = '/content/drive/MyDrive/property_data'

for path in os.listdir(dir_path):
  try:
    property_dfs.append(pd.read_csv(os.path.join(dir_path, path), low_memory=False))
    print(f'Read file: {path}')
  except:
    print(f'Error reading file: {path}')

property_data_2025 = pd.concat(property_dfs, ignore_index=True)

property_data_2025 = property_data_2025[(property_data_2025['PropertyType'] == 'Residential') & (property_data_2025['PropertySubType'] == 'SingleFamilyResidence')]

Read file: CRMLSSold202511.csv
Read file: CRMLSSold202512.csv
Read file: CRMLSSold202510.csv
Read file: CRMLSSold202509.csv
Read file: CRMLSSold202508.csv
Read file: CRMLSSold202507.csv
Read file: CRMLSSold202505.csv
Read file: CRMLSSold202506.csv
Read file: CRMLSSold202503.csv
Read file: CRMLSSold202504.csv
Read file: CRMLSSold202502.csv
Read file: CRMLSSold202501_filled.csv


In [98]:
lower_bound_close_price = property_data_2025['ClosePrice'].quantile(.005)
upper_bound_close_price = property_data_2025['ClosePrice'].quantile(.995)

property_data_2025 = property_data_2025[(property_data_2025['ClosePrice'] > lower_bound_close_price) & (property_data_2025['ClosePrice'] < upper_bound_close_price)]

In [100]:
columns_to_keep = [
    'ClosePrice', 'BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN',
    'PoolPrivateYN','OriginalListPrice', 'ListingKey', 'LivingArea', 'ListPrice', 'DaysOnMarket',
    'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'FireplacesTotal', 'AssociationFeeFrequency',
    'AssociationFee', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
    'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'YearBuilt', 'ListingId', 'City', 'TaxYear', 'NewConstructionYN',
    'BuildingAreaTotal', 'BedroomsTotal', 'BelowGradeFinishedArea', 'Stories', 'MainLevelBedrooms',
    'PostalCode', 'LotSizeSquareFeet','BathroomsTotalInteger', 'CloseDate', 'Longitude', 'Latitude',
    'SubdivisionName', 'ListingContractDate'
]
property_data_2025 = property_data_2025[columns_to_keep]

In [101]:
initial_rows = len(property_data_2025)

property_data_2025.dropna(subset=['ClosePrice'], inplace=True)
property_data_2025.drop_duplicates(inplace=True)

valid_data_mask = (
    (property_data_2025['ClosePrice'] > 0) &             # Price must be positive
    (property_data_2025['LivingArea'] > 100) &           # Living area should be realistic (>100 sqft)
    (property_data_2025['BedroomsTotal'] > 0) &          # At least 1 bedroom for Residential
    (property_data_2025['BathroomsTotalInteger'] > 0)    # Must have a bathroom
)

property_data_2025 = property_data_2025[valid_data_mask].copy()
rows_removed = initial_rows - len(property_data_2025)

print(f"Cleaning Complete.")
print(f"Rows removed due to logic errors: {rows_removed}")
print(f"Remaining clean records: {len(property_data_2025)}")

Cleaning Complete.
Rows removed due to logic errors: 185
Remaining clean records: 128720


# Handle Missing Values

In [102]:
# Dropping sparse columns
columns_to_drop = [
    'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount',
    'TaxYear', 'BelowGradeFinishedArea', 'BuilderName',
]

property_data_2025 = property_data_2025.drop(columns=columns_to_drop)

In [103]:
# Select numerical columns
numeric_cols = property_data_2025.select_dtypes(include=['number']).columns

# Fill missing values with column mean
property_data_2025[numeric_cols] = property_data_2025[numeric_cols].apply(lambda x: x.fillna(x.mean()))

# =====================================================
### BEGIN - Colin Montie
# =====================================================

# Convert Categorical Variables

In [104]:
# Binary encodes high-cardinality categorical variables
be = ce.BinaryEncoder(cols=['MLSAreaMajor', 'City', 'BuyerOfficeName', 'ListAgentFullName', 'PostalCode', 'CoListOfficeName', 'BuyerAgentAOR', 'ListAgentAOR'])
property_data_2025 = be.fit_transform(property_data_2025)

# Converts boolean variables to binary
bool_map = {True: 1, False: 0, np.nan: 0}
property_data_2025['ViewYN'] = property_data_2025['ViewYN'].map(bool_map)
property_data_2025['NewConstructionYN'] = property_data_2025['NewConstructionYN'].map(bool_map)
property_data_2025['PoolPrivateYN'] = property_data_2025['PoolPrivateYN'].map(bool_map)
property_data_2025['WaterfrontYN'] = property_data_2025['WaterfrontYN'].map(bool_map)
property_data_2025['BasementYN'] = property_data_2025['BasementYN'].map(bool_map)
property_data_2025['AttachedGarageYN'] = property_data_2025['AttachedGarageYN'].map(bool_map)

# Encodes fee frequency with a number representing the number of times per year the fee is due
fee_freq_map = {'Annually': 1, 'Monthly': 12, 'Quarterly': 4, 'SemiAnnually': 2}
property_data_2025['AssociationFeeFrequency'] = property_data_2025['AssociationFeeFrequency'].map(fee_freq_map)

# Fill NA fee frequency rows with mean
property_data_2025['AssociationFeeFrequency'] = property_data_2025['AssociationFeeFrequency'].fillna(property_data_2025['AssociationFeeFrequency'].mean())

# Creates a new df with a binary column for each flooring type
flooring_df = property_data_2025['Flooring'].str.get_dummies(sep=',').add_prefix('Flooring_')

# Concatinates flooring df to rest of data, dropping flooring column
property_data_2025 = pd.concat([property_data_2025.drop(columns=['Flooring']), flooring_df], axis=1)

In [95]:
# Confirms that categorical variables have been removed
for col in property_data_2025[list(set(property_data_2025.columns) - set(property_data_2025._get_numeric_data().columns))].columns:
  print(f'{col}: {len(property_data_2025[col].unique())}')

ListingId: 128635
CloseDate: 363
SubdivisionName: 7666


# Feature Engineering

In [105]:
# Bedroom / Bathroom Ratio
property_data_2025['BedBathRatio'] = property_data_2025['BedroomsTotal'] / property_data_2025['BathroomsTotalInteger']

# Property Age
property_data_2025['ListingContractDate'] = pd.to_datetime(property_data_2025['ListingContractDate'])
property_data_2025['PropertyAge'] = property_data_2025['ListingContractDate'].dt.year - property_data_2025['YearBuilt']

# KNN Close Price
knn = KNeighborsRegressor(n_neighbors=25,metric='haversine')
coords = np.deg2rad(property_data_2025[['Latitude', 'Longitude']])
knn.fit(coords, property_data_2025['ClosePrice'])
_, indices = knn.kneighbors(coords)
neighbor_prices = property_data_2025['ClosePrice'].values[indices[:, 1:]]
property_data_2025['NeighborsClosePrice'] = neighbor_prices.mean(axis=1)

# =====================================================
### END - Colin Montie
# =====================================================

# Create Train/Test Split

# =====================================================
### BEGIN - Leah Besser
# =====================================================

In [106]:
import pandas as pd
import numpy as np

# Convert CloseDate to datetime
property_data_2025['CloseDate'] = pd.to_datetime(property_data_2025['CloseDate'])

# Sort by date
property_data_2025 = property_data_2025.sort_values(by='CloseDate')

In [107]:
most_recent_month = property_data_2025['CloseDate'].dt.to_period('M').max()
print("Test Month:", most_recent_month)

# Create training set
train = property_data_2025[
    property_data_2025['CloseDate'].dt.to_period('M') != most_recent_month
].copy()

# Create test set (most recent month)
test = property_data_2025[
    property_data_2025['CloseDate'].dt.to_period('M') == most_recent_month
].copy()

print("Training rows:", train.shape[0])
print("Test rows:", test.shape[0])


Test Month: 2025-12
Training rows: 118391
Test rows: 10329


# Normalize Numerical Variables

In [108]:
numeric_cols = [
    'OriginalListPrice',
    'ListPrice',
    'DaysOnMarket',
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'ParkingTotal',
    'MainLevelBedrooms',
    'LotSizeSquareFeet',
    'YearBuilt',
    'NeighborsClosePrice'
]
feature_cols = [feature for feature in property_data_2025.columns if feature not in ['ClosePrice', 'ListingId', 'CloseDate', 'Latitude', 'Longitude', 'SubdivisionName', 'ListingContractDate']]

In [109]:
X_train = train[feature_cols]
y_train = train['ClosePrice']

X_test = test[feature_cols]
y_test = test['ClosePrice']


In [110]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Copy datasets
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit ONLY on training data
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Transform test using same scaler
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# =====================================================
## END - Leah Besser
# =====================================================

# Save Data

In [111]:
X_test_scaled.to_csv('property_data_test_x_engineered.csv', index=False)
X_train_scaled.to_csv('property_data_train_x_engineered.csv', index=False)
y_test.to_csv('property_data_test_y_engineered.csv', index=False)
y_train.to_csv('property_data_train_y_engineered.csv', index=False)